# Module 4 - Week 6, Topic 2: Logistic Regression
## Live Demo Notebook

**Scenario:** A Nigerian commercial bank wants to automate loan application decisions.
We build a **Logistic Regression** classifier that outputs:
1. A probability of approval (not just yes/no)
2. A decision based on a tunable threshold

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('loan_approvals.csv')
print('Shape:', df.shape)
print(f'Approval rate: {df["approved"].mean():.1%}')
df.head()

---
## Part 1 - Why Not Linear Regression for This?

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[['income', 'credit_score', 'employment_years', 'debt_ratio']]
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit a LINEAR REGRESSION on binary target
lr_wrong = LinearRegression()
lr_wrong.fit(X_train, y_train)
wrong_preds = lr_wrong.predict(X_test)

print('Linear Regression on binary target - first 10 predictions:')
print(wrong_preds[:10].round(3))
print()
print(f'Min prediction: {wrong_preds.min():.3f}  (should be 0 or more)')
print(f'Max prediction: {wrong_preds.max():.3f}  (should be 1 or less)')
print()
n_impossible = ((wrong_preds < 0) | (wrong_preds > 1)).sum()
print(f'Predictions outside [0,1]: {n_impossible} - these are impossible probabilities!')
print('This is why we need Logistic Regression.')

---
## Part 2 - The Sigmoid Function

In [ ]:
# Plot the sigmoid function
z   = np.linspace(-8, 8, 300)
sig = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(z, sig, color='steelblue', linewidth=2.5)
ax.axhline(0.5, color='grey', linestyle='--', alpha=0.7, label='50% threshold')
ax.axhline(0.0, color='black', linewidth=0.5)
ax.axhline(1.0, color='black', linewidth=0.5)

# Annotate key points
ax.annotate('z=0 → 50%', xy=(0, 0.5), xytext=(1, 0.42), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='grey'))

ax.fill_between(z, sig, 0.5, where=(sig >= 0.5), alpha=0.12,
                color='green', label='Approval zone')
ax.fill_between(z, sig, 0.5, where=(sig < 0.5), alpha=0.12,
                color='red', label='Rejection zone')

ax.set_xlabel('z (linear combination of features)', fontsize=12)
ax.set_ylabel('Predicted probability', fontsize=12)
ax.set_title('The Sigmoid Function - Maps Any Number to [0, 1]',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

print('Key property: sigmoid always stays between 0 and 1 - valid probabilities.')

---
## Part 3 - Train the Logistic Regression Model

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(random_state=42, max_iter=1000)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(f'Test accuracy: {accuracy_score(y_test, y_pred):.2%}')
print()
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))

In [ ]:
# predict_proba() returns actual probabilities
probs = pipe.predict_proba(X_test)[:, 1]  # probability of Approved

prob_df = pd.DataFrame({
    'prob_approved':  probs.round(3),
    'decision_50pct': pipe.predict(X_test),
    'actual':         y_test.values,
}).sort_values('prob_approved', ascending=False).head(12)

print('Top 12 applicants by approval probability:')
print(prob_df.to_string(index=False))
print()
print('Probabilities close to 0.5 are borderline cases - good candidates for human review.')

---
## Part 4 - Threshold Tuning

In [ ]:
# Compare three thresholds
thresholds = [0.35, 0.50, 0.70]
results = []

for t in thresholds:
    preds = (probs >= t).astype(int)
    n_approved   = preds.sum()
    true_pos     = ((preds == 1) & (y_test == 1)).sum()   # approved and repaid
    false_pos    = ((preds == 1) & (y_test == 0)).sum()   # approved but rejected
    precision    = true_pos / n_approved if n_approved > 0 else 0
    results.append((t, n_approved, true_pos, false_pos, precision))

print(f'{"Threshold":>12} {"Approved":>10} {"Correct":>10} {"Wrong":>10} {"Precision":>12}')
print('-' * 60)
for t, n_app, tp, fp, prec in results:
    print(f'{t:>12.2f} {n_app:>10} {tp:>10} {fp:>10} {prec:>11.1%}')
print()
print('Lower threshold -> more approvals, but more wrong approvals (lower precision).')
print('Higher threshold -> fewer approvals, but more reliable (higher precision).')

In [ ]:
# Precision vs Recall across thresholds
from sklearn.metrics import precision_score, recall_score

thresh_range = np.arange(0.1, 0.95, 0.05)
precisions, recalls = [], []

for t in thresh_range:
    preds = (probs >= t).astype(int)
    if preds.sum() > 0:
        precisions.append(precision_score(y_test, preds, zero_division=0))
        recalls.append(recall_score(y_test, preds, zero_division=0))
    else:
        precisions.append(0)
        recalls.append(0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresh_range, precisions, label='Precision', color='steelblue', linewidth=2)
ax.plot(thresh_range, recalls,    label='Recall',    color='coral',     linewidth=2)
ax.axvline(0.5, color='grey', linestyle='--', label='Default threshold (0.5)')
ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision vs Recall Across Thresholds\n(Higher threshold → higher precision, lower recall)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
## Part 5 - Interpret the Coefficients

In [ ]:
lr_model = pipe.named_steps['model']

coef_df = pd.DataFrame({
    'feature':     X.columns,
    'log_odds':    lr_model.coef_[0],
    'odds_ratio':  np.exp(lr_model.coef_[0]),
}).sort_values('odds_ratio', ascending=False)

print('Logistic Regression Coefficients:')
print(coef_df.round(3).to_string(index=False))
print()
print('Interpreting odds ratios:')
print('  odds_ratio > 1 : feature INCREASES approval probability')
print('  odds_ratio < 1 : feature DECREASES approval probability')
print()
for _, row in coef_df.iterrows():
    if row['odds_ratio'] > 1:
        change = (row['odds_ratio'] - 1) * 100
        print(f"  {row['feature']:<20}: 1 std increase -> approval odds * {row['odds_ratio']:.2f} (+{change:.0f}%)")
    else:
        change = (1 - row['odds_ratio']) * 100
        print(f"  {row['feature']:<20}: 1 std increase -> approval odds * {row['odds_ratio']:.2f} (-{change:.0f}%)")

---
## Part 6 - Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Rejected', 'Predicted: Approved'],
            yticklabels=['Actual: Rejected', 'Actual: Approved'],
            ax=ax, linewidths=0.5)
ax.set_title('Confusion Matrix - Loan Approval Classifier',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (rejected, would have defaulted):  {tn}  CORRECT')
print(f'False Positives (approved, actually defaulted):    {fp}  WRONG - bank loses money')
print(f'False Negatives (rejected, would have repaid):     {fn}  WRONG - bank misses revenue')
print(f'True Positives  (approved, repaid):                {tp}  CORRECT')

---
## Summary

| Concept | Key point |
|---|---|
| Why not linear regression? | Predictions outside [0,1] - impossible probabilities |
| Sigmoid function | Maps any number to a valid probability between 0 and 1 |
| `predict_proba()` | Returns the actual probability - more useful than 0/1 |
| Decision threshold | 0.5 is the default; adjust for business risk tolerance |
| Coefficients | Log-odds; e^β = odds ratio (multiplicative effect on odds) |
| Confusion matrix | Four outcomes: TP, TN, FP, FN - each has a business cost |

**Next - Topic 3:** Decision Trees. A completely different approach: instead of a mathematical function, the model asks a series of yes/no questions to classify applicants.